# DataVine Analytics
### ML Project



#### Datasets Used

1. **Wine Classification System:** (Wine dataset) - k-NN classification with PCA and hyperparameter tuning.
2. **Agricultural Feed Recommendation Engine:** (Chickwts dataset) - PCA + cosine similarity recomender.
3. **Regional Crime Pattern Analysis** (USArrests dataset) - k-Means & GMM clustering with feature selection and PCA

The notebook follows a standard workflow: data preparation → dimensionality reduction → modeling/hyperparameter tuning → evaluation & visualization.

In [ ]:
# !pip install rdatasets

In [1]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelBinarizer

# Dimentionality reduction 
from sklearn.decomposition import PCA

# Modeling
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.datasets import load_wine
from rdatasets import data

# Similarity and Evaluation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, silhouette_score



In [2]:
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42

## Step 1: Data Preparation

In [8]:
# Dataset 1: Wine for (KNN)
wine_data = load_wine()
wine = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)
wine["target"] = wine_data.target

# Dataset 2: Chickwts (for Recommendation)
chickwts = data("chickwts")

# Dataset 3: USArrests (for PCA & Clustering)
usarrests = data("USArrests").set_index("rownames")
usarrests.index.name = "State"

In [9]:
# Display dataset shapes
print("Wine dataset:        ", wine.shape)
print("Chickwts dataset:    ", chickwts.shape)
print("USAarrests dataset:  ", usarrests.shape)

Wine dataset:         (178, 14)
Chickwts dataset:     (71, 3)
USAarrests dataset:   (50, 4)


In [10]:
# Display dataset few rows
print("=== WINE ===")
print("Wine dataset:\n        ", wine.head)

print("\n=== CHICK WEIGHTS ===")
print("Chickwts dataset:\n    ", chickwts.head)

print("\n=== US ARRESTS ===")
print("USAarrests dataset:\n  ", usarrests.head)

=== WINE ===
Wine dataset:
         <bound method NDFrame.head of      alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0      14.23        1.71  2.43               15.6      127.0           2.80   
1      13.20        1.78  2.14               11.2      100.0           2.65   
2      13.16        2.36  2.67               18.6      101.0           2.80   
3      14.37        1.95  2.50               16.8      113.0           3.85   
4      13.24        2.59  2.87               21.0      118.0           2.80   
..       ...         ...   ...                ...        ...            ...   
173    13.71        5.65  2.45               20.5       95.0           1.68   
174    13.40        3.91  2.48               23.0      102.0           1.80   
175    13.27        4.28  2.26               20.0      120.0           1.59   
176    13.17        2.59  2.37               20.0      120.0           1.65   
177    14.13        4.10  2.74               24.5       96.0     

In [11]:
# Check for missing values and inconsistencies in each dataset
print("Missing values:\n")
print("Wine - ", wine.isnull().sum().sum())
print("Chickwts - ", chickwts.isnull().sum().sum())
print("USArrests - ", usarrests.isnull().sum().sum())

# Check for inconsistencies: duplicate rows, unexpected categories, out-of-range values
print("\nDuplicate rows:\n")
print("Wine - ", wine.duplicated().sum())
print("Chickwts - ", chickwts.duplicated().sum())
print("USArrests - ", usarrests.duplicated().sum())

# Check for inconsistencies: unexpected categories, out-of-range values
print("\nTarget Class & Categories:\n")
print("\nWine target classes:\n", sorted(wine['target'].unique()))
print("\nChickwts feed categories:\n", sorted(chickwts['feed'].unique()))

Missing values:

Wine -  0
Chickwts -  0
USArrests -  0

Duplicate rows:

Wine -  0
Chickwts -  0
USArrests -  0

Target Class & Categories:


Wine target classes:
 [np.int64(0), np.int64(1), np.int64(2)]

Chickwts feed categories:
 ['casein', 'horsebean', 'linseed', 'meatmeal', 'soybean', 'sunflower']


Standardizing numerical features (Z-score scaling) to improve downstream model performance

In [12]:
# Wine: standardize all numeric features columns (excluding the target)
wine_features = wine.drop(columns=["target"])
wine_scaler = StandardScaler()
wine_scaled = pd.DataFrame(
    wine_scaler.fit_transform(wine_features),
    columns=wine_features.columns
)
wine_scaled["target"] = wine["target"].values

# Chickwts: standardize the weight feature
chick_scaler = StandardScaler()
chickwts["weight_scaled"] = chick_scaler.fit_transform(chickwts[["weight"]])

# USArrests: standardize all numeric features
arrests_scaler = StandardScaler()
usarrests_scaled = pd.DataFrame(
    arrests_scaler.fit_transform(usarrests),
    columns=usarrests.columns,
    index=usarrests.index
)

print("Standardized Wine features:\n")
display(wine_scaled.head(3))

print("\nStandardized Chickwts wight:\n")
display(chickwts.head(3))

print("\nStandardized USArrests features:\n")
display(usarrests_scaled.head(3))

Standardized Wine features:



,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,1.518613,-0.562250,0.232053,-1.169593,1.913905,0.808997,1.034819,-0.659563,1.224884,0.251717,0.362177,1.847920,1.013009,0
1,0.246290,-0.499413,-0.827996,-2.490847,0.018145,0.568648,0.733629,-0.820719,-0.544721,-0.293321,0.406051,1.113449,0.965242,0
2,0.196879,0.021231,1.109334,-0.268738,0.088358,0.808997,1.215533,-0.498407,2.135968,0.269020,0.318304,0.788587,1.395148,0



Standardized Chickwts wight:



,rownames,weight,feed,weight_scaled
0,1,179,horsebean,-1.061762
1,2,160,horsebean,-1.306854
2,3,136,horsebean,-1.616444



Standardized USArrests features:



,Murder,Assault,UrbanPop,Rape
State,,,,
Alabama,1.255179,0.790787,-0.526195,-0.003451
Alaska,0.513019,1.118060,-1.224067,2.509424
Arizona,0.072361,1.493817,1.009122,1.053466


In [13]:
# Summarize the datasets to confirm structure before modeling
print("WINE Summary Statistics:\n")
display(wine.describe().T)

print("\nCHICK WEIGHTS Summary Statistics:\n")
display(chickwts[["weight"]].describe())
display(chickwts.groupby("feed")["weight"].agg(["count", "mean", "std"]))

print("\nUS-ARRESTS Summary Statistics:\n")
display(usarrests.describe().T)

WINE Summary Statistics:



,count,mean,std,min,25%,50%,75%,max
alcohol,178.0,13.000618,0.811827,11.03,12.3625,13.050,13.6775,14.83
malic_acid,178.0,2.336348,1.117146,0.74,1.6025,1.865,3.0825,5.80
ash,178.0,2.366517,0.274344,1.36,2.2100,2.360,2.5575,3.23
alcalinity_of_ash,178.0,19.494944,3.339564,10.60,17.2000,19.500,21.5000,30.00
magnesium,178.0,99.741573,14.282484,70.00,88.0000,98.000,107.0000,162.00
total_phenols,178.0,2.295112,0.625851,0.98,1.7425,2.355,2.8000,3.88
flavanoids,178.0,2.029270,0.998859,0.34,1.2050,2.135,2.8750,5.08
nonflavanoid_phenols,178.0,0.361854,0.124453,0.13,0.2700,0.340,0.4375,0.66
proanthocyanins,178.0,1.590899,0.572359,0.41,1.2500,1.555,1.9500,3.58
color_intensity,178.0,5.058090,2.318286,1.28,3.2200,4.690,6.2000,13.00



CHICK WEIGHTS Summary Statistics:



,weight
count,71.000000
mean,261.309859
std,78.073700
min,108.000000
25%,204.500000
50%,258.000000
75%,323.500000
max,423.000000


,count,mean,std
feed,,,
casein,12,323.583333,64.433840
horsebean,10,160.200000,38.625841
linseed,12,218.750000,52.235698
meatmeal,11,276.909091,64.900623
soybean,14,246.428571,54.129068
sunflower,12,328.916667,48.836384



US-ARRESTS Summary Statistics:



,count,mean,std,min,25%,50%,75%,max
Murder,50.0,7.788,4.355510,0.8,4.075,7.25,11.250,17.4
Assault,50.0,170.760,83.337661,45.0,109.000,159.00,249.000,337.0
UrbanPop,50.0,65.540,14.474763,32.0,54.500,66.00,77.750,91.0
Rape,50.0,21.232,9.366385,7.3,15.075,20.10,26.175,46.0
